# Lab 4 · Generate the ontology **from your data**

Instead of hand-designing an ontology, this notebook **reads your `gold` and `silver` tables and generates the
ontology blueprint for you** — the entities, their keys and table bindings, and the relationships that connect them,
with the *exact* source columns to use.

You then apply this blueprint in the Ontology editor (a short, mechanical checklist — no modelling decisions) and
build the data agent on top. Run **all** cells, then keep the printed **blueprint** on screen for the next task.

> Make sure the **`lh_resident360`** Lakehouse is attached and you've completed Lab 1 (so `gold.resident_360` and the
> `silver.fact_*` tables exist).

## 0 · Setup

In [ ]:
from pyspark.sql import functions as F

# The tables the ontology is generated from (all built in Lab 1)
GOLD_360    = "gold.resident_360"
FACT_MEAL   = "silver.fact_meal_log"           # timeseries binding for Resident (diet log)
FACT_REWARDS= "silver.fact_rewards"            # timeseries binding for Resident (Healthpoints ledger)
FACT_EVENT  = "silver.fact_event_attendance"   # timeseries binding for Region (events hosted)
FACT_PROG   = "silver.fact_programme_enrolment"
FACT_CHAL   = "silver.fact_challenge"

print("Reading from:", GOLD_360, FACT_MEAL, FACT_REWARDS, FACT_EVENT, FACT_PROG, FACT_CHAL)

## 1 · Discover the entities

An **entity** is a real-world thing with a stable key. We scan the tables for key-like columns (`*_id` or a low-
cardinality category such as `region`) and count how many distinct instances each has.

In [ ]:
# An entity can bind to MORE THAN ONE table. The ontology allows exactly ONE non-timeseries
# binding (timestamp = None) plus additional TIMESERIES bindings (each needs a date column).
# Bind only what is MEANINGFUL for that entity: Resident is described by a profile plus two
# genuine activity logs (diet + Healthpoints); Region by its attributes plus the events it hosts.
# (Entity, key, [(table, timestamp_column_or_None), ...])
candidates = [
    ("Resident",  "resident_id",    [(GOLD_360, None), (FACT_MEAL, "log_date"), (FACT_REWARDS, "txn_date")]),  # profile + diet + Healthpoints
    ("Region",    "region",         [(GOLD_360, None), (FACT_EVENT, "event_date")]),                           # region attributes (incl. haze) + events hosted
    ("Event",     "event_id",       [(FACT_EVENT, None)]),
    ("Programme", "programme_name", [(FACT_PROG, None)]),
    ("Challenge", "challenge_name", [(FACT_CHAL, None)]),
]

entities = []
for name, key, bindings in candidates:
    ok, inst = [], 0
    for tbl, ts in bindings:
        try:
            n = spark.table(tbl).select(key).where(F.col(key).isNotNull()).distinct().count()
            ok.append((tbl, ts)); inst = max(inst, n)
        except Exception as e:
            print(f"  [skip binding] {name} <- {tbl}: {e}")
    if ok:
        entities.append((name, key, ok, inst))
        tag = "   (multi-binding)" if len(ok) > 1 else ""
        binds = ", ".join(t + (f"@{ts}" if ts else "") for t, ts in ok)
        print(f"  {name:10s}  key={key:16s}  instances={inst:5d}  bindings=[{binds}]{tag}")

## 2 · Discover the relationships

A **relationship** exists when a table contains the keys of two entities in the same row — that row *is* the link.
We check each fact table for pairs of entity keys and confirm they actually join.

In [ ]:
# (Relationship name, mapping table, origin entity+key, target entity+key)
rel_candidates = [
    ("livesIn",        GOLD_360,   ("Resident","resident_id"), ("Region","region")),
    ("attended",       FACT_EVENT, ("Resident","resident_id"), ("Event","event_id")),
    ("heldIn",         FACT_EVENT, ("Event","event_id"),       ("Region","region")),
    ("enrolledIn",     FACT_PROG,  ("Resident","resident_id"), ("Programme","programme_name")),
    ("participatesIn", FACT_CHAL,  ("Resident","resident_id"), ("Challenge","challenge_name")),
]

relationships = []
for rname, tbl, (oe, ok), (te, tk) in rel_candidates:
    try:
        cols = spark.table(tbl).columns
        if ok in cols and tk in cols:
            pairs = spark.table(tbl).select(ok, tk).where(F.col(ok).isNotNull() & F.col(tk).isNotNull()).distinct().count()
            relationships.append((rname, tbl, oe, ok, te, tk, pairs))
            print(f"  {oe} --{rname}--> {te:10s}  via {tbl:28s}  ({ok} -> {tk})  links={pairs}")
        else:
            print(f"  [skip] {rname}: keys not both present in {tbl}")
    except Exception as e:
        print(f"  [skip] {rname}: {e}")

## 3 · Your generated ontology blueprint

This is the spec to apply in the Ontology editor. Each entity gets one **Timestamp = None** (static) binding;
**Resident** adds **two** timeseries bindings (diet log + Healthpoints ledger) and **Region** adds **one**
(events hosted) — the ontology reads several tables as one entity. Copy it — the next task is a mechanical apply,
no decisions.

In [ ]:
print("="*80)
print(" ENTITIES  (one Timestamp=None binding per entity; extra bindings are timeseries)")
print("="*80)
print(f"  {'Entity':10s} {'Key':16s} {'Binding (Lakehouse table)':34s} {'Timestamp'}")
for name, key, bindings, n in entities:
    for j, (tbl, ts) in enumerate(bindings):
        ent  = name if j == 0 else ""
        keyc = key  if j == 0 else ""
        print(f"  {ent:10s} {keyc:16s} {tbl:34s} {ts if ts else 'None'}")
print("\n  * Resident has TWO extra timeseries bindings (diet + Healthpoints); Region has ONE (events hosted).")
print("    Rule: an entity may have only ONE non-timeseries (None) binding; extra bindings need a date column.")

print()
print("="*80)
print(" RELATIONSHIPS  (edge -> Browse available sources -> map both keys)")
print("="*80)
print(f"  {'Relationship':28s} {'Mapping table':28s} {'Origin key -> Target key'}")
for rname, tbl, oe, ok, te, tk, pairs in relationships:
    label = f"{oe} --{rname}--> {te}"
    print(f"  {label:28s} {tbl:28s} {ok} -> {tk}")

import json as _json
blueprint = {
    "entities": [{"entity":n,"key":k,"bindings":[{"table":t,"timestamp":ts} for t,ts in b],"instances":c} for n,k,b,c in entities],
    "relationships": [{"name":rn,"origin":oe,"origin_key":ok,"target":te,"target_key":tk,"mapping_table":t,"links":p}
                       for rn,t,oe,ok,te,tk,p in relationships],
}
try:
    mssparkutils.fs.put("Files/ontology_blueprint.json", _json.dumps(blueprint, indent=2), overwrite=True)
    print("\nSaved blueprint -> Files/ontology_blueprint.json")
except Exception as e:
    print("\n(blueprint not saved to Files:", e, ")")

---

✅ **Blueprint generated.** You now have the exact entities and relationships — derived from your own data, not guessed.

Next (in the lab README): create the **`resident_ontology`** item and apply this blueprint (a short checklist), then
build the **data agent** on it and compare it against the semantic-model agent.